## MODELOS

In [28]:
%pip install transformers torch tqdm

You should consider upgrading via the 'c:\Python39\python.exe -m pip install --upgrade pip' command.


CONFIG

In [ ]:
from pathlib import Path
from collections import deque
from datetime import datetime as dt
import json
import re
import pprint

# Parámetros con preguntas
QUESTIONS = {
    "precio": "¿Cuál es el precio del producto?",
    "nombre": "¿Cuál es el nombre del producto?",
    "marca": "¿Cuál es la marca del producto?",
    "unidad": "¿En qué unidad se vende el producto?",
    "precio_unidad_basica": "¿Cuál es el precio por unidad básica?",
}

MAX_ATTEMPTS = 5
NEED_MATCHES = 2

CARGA DEL MODELO

In [30]:
from transformers import pipeline, AutoTokenizer, AutoModelForQuestionAnswering

MODEL_ID = "PlanTL-GOB-ES/roberta-base-bne-sqac"

def load_qa_model(model_id: str = MODEL_ID):
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    model = AutoModelForQuestionAnswering.from_pretrained(model_id, device_map="auto")
    return pipeline("question-answering", model=model, tokenizer=tokenizer)

qa_pipe = load_qa_model()

Device set to use cpu


LIMPIEZA

In [31]:
import html as html_lib

TAG_RE = re.compile(r"<[^>]+>")
SCRIPT_RE = re.compile(r"<(script|style).*?>.*?</\1>", re.I | re.S)
WS_RE = re.compile(r"\s+")
BODY_RE = re.compile(r"<body[^>]*>(.*?)</body>", re.I | re.S)

def extract_body_content(raw_html: str):
    match = BODY_RE.search(raw_html)
    return match.group(1).strip() if match else raw_html

def strip_html(raw: str) -> str:
    raw = SCRIPT_RE.sub(" ", raw)
    raw = TAG_RE.sub(" ", raw)
    return WS_RE.sub(" ", html_lib.unescape(raw)).strip()

PREGUNTA

In [32]:
def ask_model(context: str):
    """Devuelve un dict con todas las respuestas de QA"""
    respuestas = {}
    for key, pregunta in QUESTIONS.items():
        result = qa_pipe(question=pregunta, context=context)
        answer = result.get("answer", "").strip()
        if not answer or answer.lower() in {"no", "ninguno"}:
            answer = "NA"
        respuestas[key] = answer
        
    if any(value == "NA" for value in respuestas.values()):
        return None
    return respuestas

FORMATEAR (HTML - TXT)

In [ ]:
text_path = Path("data/processed/Brazil/1_extracted.txt")
html_path = Path("data/raw/Mexico/Alsuper/Arroz/Verde Valle/1.html")

def procesar_archivos_reales(url:str, retail: str, country: str):
    resultado_final = {
        "HTML": {},
        "TXT": {}
    }

    # 📥 Procesar HTML
    if html_path.exists():
        raw_html = html_path.read_text(errors="ignore")
        html_content = extract_body_content(raw_html)
        context = strip_html(html_content)

        respuestas = deque(maxlen=NEED_MATCHES)
        for intento in range(1, MAX_ATTEMPTS + 1):
            result = ask_model(context)
            if not result:
                print(f"❌ HTML intento {intento} fallido.")
                continue

            result["url"] = url
            result["retail"] = retail
            result["pais"] = country
            respuestas.append(json.dumps(result, sort_keys=True))

            if len(respuestas) == NEED_MATCHES and len(set(respuestas)) == 1:
                print(f"✅ HTML: respuesta estable (intento {intento})")
                key = result["nombre"].lower().replace(" ", "_")
                resultado_final["HTML"][key] = result
                break
    else:
        print("❌ HTML no encontrado:", html_path)

    # 📝 Procesar TXT
    if text_path.exists():
        context = text_path.read_text(errors="ignore").strip()

        respuestas = deque(maxlen=NEED_MATCHES)
        for intento in range(1, MAX_ATTEMPTS + 1):
            result = ask_model(context)
            if not result:
                print(f"❌ TXT intento {intento} fallido.")
                continue

            result["url"] = url
            result["retail"] = retail
            result["pais"] = country
            respuestas.append(json.dumps(result, sort_keys=True))

            if len(respuestas) == NEED_MATCHES and len(set(respuestas)) == 1:
                print(f"✅ TXT: respuesta estable (intento {intento})")
                key = result["nombre"].lower().replace(" ", "_")
                resultado_final["TXT"][key] = result
                break
    else:
        print("❌ TXT no encontrado:", text_path)

    # Mostrar resultados
    print("\n🔍 Resultado final:\n")
    pprint.pprint(resultado_final)

    return resultado_final

HTML (o texto)

In [ ]:
productos = procesar_archivos_reales(url= "Esto es una url", retail="Alsuper", country="Mexico")

✅ HTML: respuesta estable (intento 2)
✅ TXT: respuesta estable (intento 2)

🔍 Resultado final:

{'HTML': {'arroz_salvaje': {'marca': 'Arroz Marca Verde Valle',
                            'nombre': 'Arroz Salvaje',
                            'pais': 'Mexico',
                            'precio': '252246 $38.90',
                            'precio_unidad_basica': '252246 $38.90',
                            'retail': 'Alsuper',
                            'unidad': '252246 $38.90',
                            'url': '252246'}},
 'TXT': {'arroz_súper_extra_verde_valle': {'marca': 'Verde Valle',
                                           'nombre': 'Arroz súper extra Verde '
                                                     'Valle',
                                           'pais': 'Mexico',
                                           'precio': '$31.90',
                                           'precio_unidad_basica': '252246\n'
                                                     